In [24]:
import pandas as pd

print("Loading 1M+ rows of GitHub data... this might take a few seconds.")
# 1. Load the raw data. We tell pandas there is no header row, and name the columns.
df = pd.read_csv("github_commits_raw.csv", names=["repo", "date"])

# 2. Clean the duplicates (This fixes the overlapping data from when the Wi-Fi dropped)
original_count = len(df)
df = df.drop_duplicates()
print(f"Removed {original_count - len(df)} duplicate rows.")

# 3. Group the data by Date to get the total daily activity across all 50 repos
daily_activity = df.groupby('date').size().reset_index(name='github_commits')

# 4. Save the clean, compressed version
daily_activity.to_csv("github_daily_summary.csv", index=False)
print(f"Success! Data compressed down to {len(daily_activity)} daily rows.")

Loading 1M+ rows of GitHub data... this might take a few seconds.
Removed 1003834 duplicate rows.
Success! Data compressed down to 2073 daily rows.


In [25]:
# Filter the data to only include target years (2020 - 2023)
df_filtered = daily_activity[(daily_activity['date'] >= '2020-01-01') & (daily_activity['date'] <= '2023-12-31')]

# Save the trimmed file
df_filtered.to_csv("github_daily_summary_clean.csv", index=False)

print(f"Perfect! We now have exactly {len(df_filtered)} days of data.")

Perfect! We now have exactly 1461 days of data.


In [26]:
import pandas as pd

print("Loading the three pillars...")

# 1. Loading the cleaned GitHub data
github_df = pd.read_csv("github_daily_summary_clean.csv")
github_df['date'] = pd.to_datetime(github_df['date']).dt.strftime('%Y-%m-%d')

# 2. Loading and cleaning the NASDAQ data
nasdaq_df = pd.read_csv("nasdaq_data.csv")
nasdaq_df['date'] = pd.to_datetime(nasdaq_df['date'], utc=True).dt.strftime('%Y-%m-%d')

# 3. Loading and cleaning the Layoffs data
layoffs_df = pd.read_csv("tech_winter_2020_to_2023.csv", header=None)
layoffs_df = layoffs_df.rename(columns={2: 'layoffs_count', 3: 'date'})

# Telling pandas to force bad text into blanks, then drop the blanks 
# (Added format='mixed' to cleanly handle date parsing without warnings)
layoffs_df['date'] = pd.to_datetime(layoffs_df['date'], format='mixed', errors='coerce')
layoffs_df = layoffs_df.dropna(subset=['date'])
layoffs_df['date'] = layoffs_df['date'].dt.strftime('%Y-%m-%d')

# Turning the word "Laid_Off_Count" into a blank, then a 0
layoffs_df['layoffs_count'] = pd.to_numeric(layoffs_df['layoffs_count'], errors='coerce').fillna(0)

# Grouping layoffs by day
daily_layoffs = layoffs_df.groupby('date')['layoffs_count'].sum().reset_index()

print("Merging them together...")

master_df = pd.merge(github_df, nasdaq_df, on='date', how='left')
master_df = pd.merge(master_df, daily_layoffs, on='date', how='left')

# Filling any days with no layoffs with a 0
master_df['layoffs_count'] = master_df['layoffs_count'].fillna(0)

# Saving the final master dataset
master_df.to_csv("FINAL_MASTER_DATASET.csv", index=False)

print(f"Success! Master dataset created with {len(master_df)} daily rows.")
display(master_df.head(10))

Loading the three pillars...
Merging them together...
Success! Master dataset created with 1461 daily rows.


,date,github_commits,nasdaq_price,layoffs_count
0,2020-01-01,25,NaN,0.0
1,2020-01-02,36,8872.219727,0.0
2,2020-01-03,36,8793.900391,0.0
3,2020-01-04,32,NaN,0.0
4,2020-01-05,24,NaN,0.0
5,2020-01-06,36,8848.519531,0.0
6,2020-01-07,35,8846.450195,0.0
7,2020-01-08,38,8912.370117,0.0
8,2020-01-09,37,8989.629883,0.0
9,2020-01-10,35,8966.639648,0.0
